# verificar la existencia de las estrellas Gran1 

In [22]:
import pandas as pd
from astropy.coordinates import SkyCoord
import astropy.units as u

csv_felipe = "/home/nacho/molecfit_test/gran01.csv"
df_tabla = pd.read_csv(csv_felipe)

csv_mio = "/home/nacho/molecfit_test/coordenadas_convertidas.csv"
df_mio = pd.read_csv(csv_mio)

# Crear SkyCoord de las coordenadas de mi tabla
coords_mio = SkyCoord(
    ra=df_mio['RA_deg'].values * u.deg, 
    dec=df_mio['DEC_deg'].values * u.deg, 
    frame='icrs'
)

# Crear SkyCoord del catálogo de Felipe
coord_felipe = SkyCoord(
    ra=df_tabla['RA_deg'].values * u.deg, 
    dec=df_tabla['Dec_deg'].values * u.deg, 
    frame='icrs'
)

# Buscar las coincidencias más cercanas
idx, sep2d, _ = coords_mio.match_to_catalog_sky(coord_felipe)

# Agregar los resultados al DataFrame
df_final = df_mio.copy()
df_final['GC_ID'] = df_tabla.iloc[idx]['GC_ID'].values
df_final['sep_arcsec'] = sep2d.arcsec
df_final['Teff_felipe'] = df_tabla.iloc[idx]['Teff_K'].values
df_final['ID_felipe'] = df_tabla.iloc[idx]['GC_ID'].values

df_final


,ID,RA_deg,DEC_deg,GC_ID,sep_arcsec,Teff_felipe,ID_felipe
0,estrella1,269.655208,-32.018417,Gran 01-090,2.978337,4358,Gran 01-090
1,estrella3,269.650458,-32.019056,Gran 01-087,5.031583,4477,Gran 01-087
2,estrella4,269.653792,-32.020222,Gran 01-083,2.943944,3994,Gran 01-083
3,estrella7,269.660333,-32.016278,Gran 01-106,5.181825,4019,Gran 01-106
4,estrella8,269.659667,-32.022611,Gran 01-027,3.962254,4577,Gran 01-027
5,estrella9,269.653167,-32.011556,Gran 01-140,6.304822,4184,Gran 01-140


# leer el archivo de topcat


In [23]:
from astropy.table import Table

# ver resultados
ruta_gaiaa2mass = "/home/nacho/molecfit_test/gran01_gaia_2mass_grande.fits"
fits_gaia2mass = Table.read(ruta_gaiaa2mass)
df_gaia2mass = fits_gaia2mass.to_pandas()
df_gaia2mass.head()

,_RAJ2000_1,_DEJ2000_1,_r_1,RA_ICRS,DE_ICRS,Source,e_RA_ICRS,e_DE_ICRS,Plx,e_Plx,...,e_Hmag,Kmag,e_Kmag,Qflg,Rflg,Bflg,Cflg,Xflg,Aflg,Separation
0,269.653782,-32.109899,0.089965,269.653784,-32.109935,4043748577184188544,0.0696,0.0550,-0.1159,0.0786,...,0.023,9.314,0.021,b'AAA',b'222',b'111',b'000',0,0,0.027802
1,269.662712,-32.107538,0.088111,269.662710,-32.107550,4043748577185171072,0.0646,0.0511,0.4133,0.0709,...,NaN,13.625,NaN,b'BUU',b'200',b'100',b'c00',0,0,0.047913
2,269.661491,-32.106486,0.086969,269.661470,-32.106515,4043748577185372544,0.1557,0.1251,-0.2223,0.1606,...,0.073,13.187,0.059,b'AAA',b'222',b'111',b'0c0',0,0,0.388420
3,269.659859,-32.108165,0.088518,269.659835,-32.108200,4043748577219262336,0.1067,0.0886,0.1689,0.1128,...,0.065,14.045,NaN,b'UBU',b'20',b'10',b'000',0,0,0.130592
4,269.654334,-32.106490,0.086560,269.654294,-32.106515,4043748577316820096,0.1282,0.1024,0.1788,0.1369,...,0.062,11.534,0.050,b'AAA',b'222',b'222',b'ccc',0,0,0.118038


In [24]:
from astropy.table import Table 
from astropy.coordinates import SkyCoord
from astropy import units as u
import numpy as np
import pandas as pd

# Leer tablas
ruta = "/home/nacho/molecfit_test/gran01_gaia_2mass_grande.fits"
tabla = Table.read(ruta)
df_tabla = tabla.to_pandas()


df_coordenadas = df_final

print(f"Estrellas observadas: {len(df_coordenadas)}")
print(f"Estrellas en catálogo Gaia+2MASS: {len(df_tabla)}")

# Crear SkyCoord de tus estrellas observadas
coords_observadas = SkyCoord(
    ra=df_coordenadas['RA_deg'].values * u.deg, 
    dec=df_coordenadas['DEC_deg'].values * u.deg, 
    frame='icrs'
)

# Crear SkyCoord del catálogo (verifica el nombre exacto de las columnas)

coords_catalogo = SkyCoord(
    ra=df_tabla['RA_ICRS'].values * u.deg, 
    dec=df_tabla['DE_ICRS'].values * u.deg, 
    frame='icrs'
)

# Hacer el match: buscar para cada estrella observada la más cercana en el catálogo
idx_match, sep2d, dist3d = coords_observadas.match_to_catalog_sky(coords_catalogo)

# Convertir separación a arcsec
sep_arcsec = sep2d.to(u.arcsec).value

# Crear DataFrame con resultados
resultados = []

for i, (estrella_id, sep) in enumerate(zip(df_coordenadas.index, sep_arcsec)):
    if sep < 7.0:  # Match exitoso si está a menos de 7 arcsec
        # Obtener datos del catálogo
        match_idx = idx_match[i]
        datos_catalogo = df_tabla.iloc[match_idx]
        
        resultado = {
            'ID': estrella_id,
            'ID_felipe': df_coordenadas.loc[estrella_id, 'ID_felipe'],
            'RA_obs': df_coordenadas.loc[estrella_id, 'RA_deg'],
            'DEC_obs': df_coordenadas.loc[estrella_id, 'DEC_deg'],
            'RA_cat': datos_catalogo['_RAJ2000_1'],
            'DEC_cat': datos_catalogo['_DEJ2000_1'],
            'sep_arcsec': sep,
            'Teff_felpe': df_coordenadas.loc[estrella_id, 'Teff_felipe'],
            'Gmag': datos_catalogo.get('Gmag', np.nan),
            'BPmag': datos_catalogo.get('BPmag', np.nan),
            'RPmag': datos_catalogo.get('RPmag', np.nan),
            'BP-RP': datos_catalogo.get('BP-RP', np.nan),
            'Jmag': datos_catalogo.get('Jmag', np.nan),
            'Hmag': datos_catalogo.get('Hmag', np.nan),
            'Kmag': datos_catalogo.get('Kmag', np.nan),
            'e_Jmag': datos_catalogo.get('e_Jmag', np.nan),
            'e_Hmag': datos_catalogo.get('e_Hmag', np.nan),
            'e_Kmag': datos_catalogo.get('e_Kmag', np.nan),
        }
        resultados.append(resultado)
        
        print(f"{estrella_id}: Match encontrado (sep = {sep:.3f} arcsec)")
        print(f"  J={datos_catalogo.get('Jmag', 'N/A'):.3f}, "
              f"H={datos_catalogo.get('Hmag', 'N/A'):.3f}, "
              f"K={datos_catalogo.get('Kmag', 'N/A'):.3f}")
    else:
        print(f"{estrella_id}: NO encontrado (sep = {sep:.3f} arcsec)")
        resultados.append({
            'ID': estrella_id,
            'RA_obs': df_coordenadas.loc[estrella_id, 'RA_deg'],
            'DEC_obs': df_coordenadas.loc[estrella_id, 'DEC_deg'],
            'sep_arcsec': sep,
        })

# Crear DataFrame final
df_final = pd.DataFrame(resultados)

print(f"\n{'='*60}")
print(f"Matches exitosos: {(df_final['sep_arcsec'] < 7.0).sum()} / {len(df_coordenadas)}")
print(f"{'='*60}")

# Mostrar resultados
df_final

Estrellas observadas: 6
Estrellas en catálogo Gaia+2MASS: 2926
0: Match encontrado (sep = 5.005 arcsec)
  J=13.292, H=12.351, K=12.037
1: Match encontrado (sep = 4.692 arcsec)
  J=12.300, H=11.377, K=11.084
2: Match encontrado (sep = 2.853 arcsec)
  J=13.292, H=12.351, K=12.037
3: Match encontrado (sep = 4.704 arcsec)
  J=13.903, H=12.921, K=12.679
4: Match encontrado (sep = 4.713 arcsec)
  J=12.060, H=10.993, K=10.720
5: Match encontrado (sep = 6.280 arcsec)
  J=13.580, H=12.654, K=12.426

Matches exitosos: 6 / 6


,ID,ID_felipe,RA_obs,DEC_obs,RA_cat,DEC_cat,sep_arcsec,Teff_felpe,Gmag,BPmag,RPmag,BP-RP,Jmag,Hmag,Kmag,e_Jmag,e_Hmag,e_Kmag
0,0,Gran 01-090,269.655208,-32.018417,269.654227,-32.019468,5.004957,4358,16.468254,17.841732,15.260636,2.581096,13.292,12.351,12.037,0.061,0.077,0.048
1,1,Gran 01-087,269.650458,-32.019056,269.649243,-32.019775,4.692210,4477,16.156706,16.683660,14.226932,2.456729,12.300,11.377,11.084,0.072,0.089,0.061
2,2,Gran 01-083,269.653792,-32.020222,269.654227,-32.019468,2.852801,3994,16.468254,17.841732,15.260636,2.581096,13.292,12.351,12.037,0.061,0.077,0.048
3,3,Gran 01-106,269.660333,-32.016278,269.661792,-32.015790,4.703619,4019,16.999964,18.490936,15.853140,2.637796,13.903,12.921,12.679,0.083,0.083,0.065
4,4,Gran 01-027,269.659667,-32.022611,269.658326,-32.023156,4.712858,4577,15.505630,16.852260,14.150039,2.702222,12.060,10.993,10.720,0.041,0.038,0.033
5,5,Gran 01-140,269.653167,-32.011556,269.651824,-32.012812,6.280222,4184,16.493430,17.788250,15.361537,2.426713,13.580,12.654,12.426,0.047,0.052,0.039


# relacion color temperatura



In [25]:
A_V = 3.38 # segun el paper de felipe

A_bp = 1.08337*A_V
A_rp = 0.63439*A_V# estos segun el paper Sedanur İyisan et al 2025 AJ 169 138

df_2 = df_final.copy()
df_2['BPmag_0'] = df_2['BPmag'] - A_bp
df_2['RPmag_0'] = df_2['RPmag'] - A_rp
df_2['BP_RP0']= df_2['BPmag_0'] - df_2['RPmag_0']


In [26]:
min_color = df_2['BP_RP0'].min()
max_color = df_2['BP_RP0'].max()
print(min_color, max_color)



0.9091606000000017 1.184668600000002


# calculo de la temperatura efectiva

In [27]:
import numpy as np

x = df_2['BP_RP0'].values  # color corregido
df_2['Teff_BP_RP'] = 7981 - 4138.3*x + 1264.9366*x**2 -103.4388*x**3 # ecuacion casagrande et all 2021
df_2['Teff_BP_RP']
df_2

,ID,ID_felipe,RA_obs,DEC_obs,RA_cat,DEC_cat,sep_arcsec,Teff_felpe,Gmag,BPmag,...,Jmag,Hmag,Kmag,e_Jmag,e_Hmag,e_Kmag,BPmag_0,RPmag_0,BP_RP0,Teff_BP_RP
0,0,Gran 01-090,269.655208,-32.018417,269.654227,-32.019468,5.004957,4358,16.468254,17.841732,...,13.292,12.351,12.037,0.061,0.077,0.048,14.179941,13.116398,1.063544,4886.101965
1,1,Gran 01-087,269.650458,-32.019056,269.649243,-32.019775,4.692210,4477,16.156706,16.683660,...,12.300,11.377,11.084,0.072,0.089,0.061,13.021869,12.082694,0.939176,5124.459201
2,2,Gran 01-083,269.653792,-32.020222,269.654227,-32.019468,2.852801,3994,16.468254,17.841732,...,13.292,12.351,12.037,0.061,0.077,0.048,14.179941,13.116398,1.063544,4886.101965
3,3,Gran 01-106,269.660333,-32.016278,269.661792,-32.015790,4.703619,4019,16.999964,18.490936,...,13.903,12.921,12.679,0.083,0.083,0.065,14.829145,13.708902,1.120244,4787.103776
4,4,Gran 01-027,269.659667,-32.022611,269.658326,-32.023156,4.712858,4577,15.505630,16.852260,...,12.060,10.993,10.720,0.041,0.038,0.033,13.190469,12.005801,1.184669,4681.769685
5,5,Gran 01-140,269.653167,-32.011556,269.651824,-32.012812,6.280222,4184,16.493430,17.788250,...,13.580,12.654,12.426,0.047,0.052,0.039,14.126459,13.217299,0.909161,5186.450149


In [28]:
# calcular diferencias entre temperaturas 
df_2['▲Teff'] = abs(df_2['Teff_felpe'] - df_2['Teff_BP_RP'])  

# buscar estrellas que se parezcan mas en temperatura
ruta = "/home/nacho/molecfit_test/gran01.csv"
df_temperaturas = pd.read_csv(ruta)
temps = df_temperaturas['Teff_K'].values
temps_mias = df_2['Teff_BP_RP'].values

# eligir las que tengan menos diferencia de temperatura

diferencias = np.abs(temps[:, np.newaxis] - temps_mias[np.newaxis, :])
indices_minimos = np.argmin(diferencias, axis=0)
print(indices_minimos)    
estrellas_similares = temps[indices_minimos]
df_2['Teff_similar_felipe'] = estrellas_similares
df_2['▲Teff_similar'] = df_2['Teff_felpe'] - df_2['Teff_similar_felipe']

# entregar las estrellas similares:
df_temperaturas_similares = df_temperaturas.iloc[indices_minimos].reset_index(drop=True)
df_temperaturas_similares

[16 16 16 16 35 16]


,GC_ID,RA_deg,Dec_deg,Teff_K,logg_dex,FeH_dex,RV_kms,SN
0,Gran 01-061,269.64709,-32.02050,4776,1.8,-1.02,77.03,79
1,Gran 01-061,269.64709,-32.02050,4776,1.8,-1.02,77.03,79
2,Gran 01-061,269.64709,-32.02050,4776,1.8,-1.02,77.03,79
3,Gran 01-061,269.64709,-32.02050,4776,1.8,-1.02,77.03,79
4,Gran 01-118,269.65076,-32.01565,4664,1.3,-1.08,75.08,161
5,Gran 01-061,269.64709,-32.02050,4776,1.8,-1.02,77.03,79


# calculo de la correccion bolometrica y la magnitud bolometrica 


In [31]:
import gdr3bcg.bcg as bcg

feh_inicial = -1.17
logg_inicial = 1.5
alpha_fe = 0.0 

tabla_bc = bcg.BolometryTable()
resultados_bc_gaia = []

for i, col in df_2.iterrows():
    ids = col['ID']
    teff = col['Teff_colte']
    teff_mio = col['Teff_BP_RP']
    try:
        # Punto en el espacio de parámetros: [Teff, logg, [Fe/H], [alpha/Fe]]
        punto = [teff, logg_inicial, feh_inicial, alpha_fe]
        punto_mio = [teff_mio, logg_inicial, feh_inicial, alpha_fe]
        # Calcular BC_G
        # offset=0.0 por defecto (BC_G,Sun = +0.08 mag)
        bc_g = tabla_bc.computeBc(punto, offset=0.0)
        bc_g_mio = tabla_bc.computeBc(punto_mio, offset=0.0)
        
        print(f' {ids}: Teff={teff:.0f} K --> BC_G={bc_g:.3f} mag')
        
        resultados_bc_gaia.append({
            'ID': ids,
            'Teff': teff,
            'BC_G_gaia': bc_g,
            'BC_G_mio': bc_g_mio,
            'error': None
        })
        
    except Exception as e:
        print(f'x {ids}: Error al calcular BC_G - {e}')
        resultados_bc_gaia.append({
            'ID': ids,
            'Teff': teff,
            'BC_G_gaia': np.nan,
            'BC_G_mio': np.nan,
            'error': str(e)
        })
# ahora el calculo de la magnitud bolometrica 
bc_gaia_list = [r['BC_G_gaia'] for r in resultados_bc_gaia]
bc_gaia_mio = [r['BC_G_mio'] for r in resultados_bc_gaia]
df_2['BC_G_gaia'] = bc_gaia_list
df_2['BC_G_mio'] = bc_gaia_mio


d = 7940 # distancia en parsecs 
# ecuacion modulo distancia 
mu = 5*np.log10(d) - 5

# ecuacion magnitud bolometrica Mbol = Mabs + BC
df_2['Mbol_gaia'] = df_2['Gmag'] - mu + df_2['BC_G_gaia']
df_2['Mbol_mio'] = df_2['Gmag'] - mu + df_2['BC_G_mio']
df_2



 0: Teff=3950 K --> BC_G=-0.485 mag
 1: Teff=3889 K --> BC_G=-0.530 mag
 2: Teff=3950 K --> BC_G=-0.485 mag
 3: Teff=3951 K --> BC_G=-0.484 mag
 4: Teff=3730 K --> BC_G=-0.653 mag
 5: Teff=4224 K --> BC_G=-0.312 mag


/home/nacho/.conda/envs/molecfit/lib/python3.12/site-packages/gdr3bcg/bcg.py:43: FutureWarning: The 'delim_whitespace' keyword in pd.read_table is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  table=pd.read_table(file, delim_whitespace=True, names=["teff", "logg","metal","alpha","bc"],dtype={'teff':np.float64,'logg':np.float64,'metal':np.float64,'alpha':np.float64,'bc':np.float64})


,ID,ID_felipe,RA_obs,DEC_obs,RA_cat,DEC_cat,sep_arcsec,Teff_felpe,Gmag,BPmag,...,BP_RP0,Teff_BP_RP,▲Teff,Teff_similar_felipe,▲Teff_similar,Teff_colte,BC_G_gaia,BC_G_mio,Mbol_gaia,Mbol_mio
0,0,Gran 01-090,269.655208,-32.018417,269.654227,-32.019468,5.004957,4358,16.468254,17.841732,...,1.063544,4886.101965,528.101965,4776,-418,3949.500000,-0.485387,-0.066076,1.483765,1.903076
1,1,Gran 01-087,269.650458,-32.019056,269.649243,-32.019775,4.692210,4477,16.156706,16.683660,...,0.939176,5124.459201,647.459201,4776,-299,3889.200000,-0.529957,-0.016524,1.127646,1.641080
2,2,Gran 01-083,269.653792,-32.020222,269.654227,-32.019468,2.852801,3994,16.468254,17.841732,...,1.063544,4886.101965,892.101965,4776,-782,3949.500000,-0.485387,-0.066076,1.483765,1.903076
3,3,Gran 01-106,269.660333,-32.016278,269.661792,-32.015790,4.703619,4019,16.999964,18.490936,...,1.120244,4787.103776,768.103776,4776,-757,3951.000000,-0.484307,-0.091011,2.016555,2.409851
4,4,Gran 01-027,269.659667,-32.022611,269.658326,-32.023156,4.712858,4577,15.505630,16.852260,...,1.184669,4681.769685,104.769685,4664,-87,3730.500000,-0.653364,-0.121529,0.353163,0.884998
5,5,Gran 01-140,269.653167,-32.011556,269.651824,-32.012812,6.280222,4184,16.493430,17.788250,...,0.909161,5186.450149,1002.450149,4776,-592,4224.416667,-0.311855,-0.006268,1.682472,1.988060


# calcular la log g 


In [32]:
M_sun = 1.989e30  # kg
M_bol_sun = 4.74  # magnitud bolometrica del Sol en V
T_sun = 5770 # temperatura efectiva del Sol en K
logg = 4.44 # logg del sol
M_star = 0.8*M_sun
logg_list = []
logg_mio = []
# calcular la magnitud bolometrica de la estrella
for idx, row in df_2.iterrows():
    id_s = row['ID']
    teff = row['Teff_colte']  # Usar Teff de colte (más preciso)
    teff_mio = row['Teff_BP_RP'] # Usar mi ecuacion
    mbol = row['Mbol_gaia']
    mbol_mio = row['Mbol_mio']
    
    if pd.isna(teff) or pd.isna(mbol):
        print(f"✗ {id_s}: Faltan datos (Teff o Mbol)")
        logg_list.append(np.nan)
        logg_mio.append(np.nan)
        continue
    log_g_estrella = (logg+ 
                      4 * np.log10(teff / T_sun) + 
                      0.4 * (mbol - M_bol_sun) + 
                      np.log10(M_star / M_sun))
    log_g_mio = (logg+ 
                      4 * np.log10(teff_mio / T_sun) + 
                      0.4 * (mbol - M_bol_sun) + 
                      np.log10(M_star / M_sun))
    
    logg_list.append(log_g_estrella)
    logg_mio.append(log_g_mio)
    
df_2['logg_colte'] = logg_list
df_2['logg_mio'] = logg_mio

df_2.to_csv('/home/nacho/molecfit_test/datos_fotometricos_finales.csv', index=False)

In [33]:
# vamos a tomar los datos de una estrella no mas
rederin = df_2['BP_RP0'][0]
feh_ = -1.21
Gmag = df_final['Gmag'][0]
BPmag = df_final['BPmag'][0]
RPmag = df_final['RPmag'][0]
mag_j = df_final['Jmag'][0]
mag_h = df_final['Hmag'][0]
mag_k = df_final['Kmag'][0]
print(rederin, feh_, Gmag, BPmag, RPmag, mag_j, mag_h, mag_k)


1.063543600000001 -1.21 16.468254 17.841732 15.260636 13.292 12.351 12.037


# colte

In [34]:
# usar colte 

import sys
sys.path.append('/home/nacho/molecfit_test/colte')  
from colte import colte
a = 0 
teff = []
while a< len(df_final):
    rederin = df_2['BP_RP0'][a]
    feh_ = -1.21 # metalicidad de los papers
    Gmag = df_final['Gmag'][a]
    BPmag = df_final['BPmag'][a]
    RPmag = df_final['RPmag'][a]
    mag_j = df_final['Jmag'][a]
    mag_h = df_final['Hmag'][a]
    mag_k = df_final['Kmag'][a]

    logg = 1.1 # gravedad obtenida de los papers
    sid = f'star{a+1}'
    ebv = 0.8785 # reddening obtenido del la tool de ipac/nasa
    feh = feh_
    gg = Gmag
    bp = BPmag
    rp = RPmag
    j2 = mag_j
    h2 = mag_h
    k2 = mag_k
    colte(sid,logg,feh,gg,bp,rp,j2,h2,k2,
        ebv,DR2=False,DR3=True,bprp_ex=None,pmod=None,
        COD=False,outfile=False,MC=False,trials=False,wato=False,
        elogg=None,efeh=None,egg=None,ebp=None,erp=None,ej2=None,eh2=None,
        ek2=None,eebv=None)
    ruta_csv_colte = "/home/nacho/molecfit_test/colte.csv"
    df_colte = pd.read_csv(ruta_csv_colte)
    T_cols = [col for col in df_colte.columns if col.startswith('T_')]
    teff_star = df_colte[T_cols].mean().mean()
    teff.append(teff_star)
    a+=1
# hacer lo mismo para todas las estrellas:
teff
df_2['Teff_colte'] = teff
df_2[['Teff_BP_RP','Teff_colte']]


,Teff_BP_RP,Teff_colte
0,4886.101965,3949.500000
1,5124.459201,3889.200000
2,4886.101965,3949.500000
3,4787.103776,3951.000000
4,4681.769685,3730.500000
5,5186.450149,4224.416667


In [35]:
# ver cual estrella del paper de felipe se parece mas a la de colte en temperatura
import numpy as np

temperaturass_colte = df_2['Teff_colte'].values
temps_felipe = df_temperaturas['Teff_K'].values

# eligir las que tengan menos diferencia de temperatura
diferencias = np.abs(temps_felipe[:, np.newaxis] - temperaturass_colte[np.newaxis, :])
indices_minimos = np.argmin(diferencias, axis=0)
print(indices_minimos)
# extraer id de las estrellas similares
estrellas_similares_felipe = df_temperaturas.iloc[indices_minimos].reset_index(drop=True)
teff_sim_a_colte = estrellas_similares_felipe['Teff_K'].values
df_2['delta_T_colte_felipe'] = df_2['Teff_colte'] - teff_sim_a_colte
df_2[['delta_T_colte_felipe']]

[21 21 21 21 21 30]


,delta_T_colte_felipe
0,-44.500000
1,-104.800000
2,-44.500000
3,-43.000000
4,-263.500000
5,-8.583333
